# HW3 Part 2




**Course Code:**

**Group number:**

**Student Name:**

**Student ID:**

# Turn Continuity Classification


**Task**: Binary classification — predict whether a spoken turn is **Complete (1)** or **Incomplete (0)**.

**Metric**: Macro-F1 Score.

| Label | Meaning |
|---|---|
| 1 | **Complete** — semantic intent is finished; system can respond |
| 0 | **Incomplete** — intent is unfinished; system should keep listening |


## 0. Environment Setup & Data Loading

In [4]:
# Phase 2 only requires the standard scientific Python stack:
# pandas, numpy, scikit-learn. These are typically pre-installed.
# Uncomment the next line if you need to install them in a clean environment.
# !pip install -q pandas numpy scikit-learn


In [5]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.utils import resample

print("All imports successful!")


All imports successful!


In [6]:
# Resolve train.csv / test.csv whether the notebook runs from part2/ or repo root.
def find_csv(name: str) -> Path:
    candidates = [Path(name), Path('part2') / name, Path('..') / name]
    for c in candidates:
        if c.exists():
            return c
    raise FileNotFoundError(
        f"Could not locate {name}. Tried: {[str(p) for p in candidates]}"
    )

train_path = find_csv('train.csv')
test_path = find_csv('test.csv')
print(f"train: {train_path.resolve()}")
print(f"test : {test_path.resolve()}")

train_df = pd.read_csv(train_path)
public_test_df = pd.read_csv(test_path)

print(f"\nTrain size: {len(train_df)}  cols: {list(train_df.columns)}")
print(f"Test  size: {len(public_test_df)}  cols: {list(public_test_df.columns)}")
if 'label' in train_df.columns:
    print('\nLabel distribution:')
    print(train_df['label'].value_counts())
display(train_df.head())


train: C:\Coding\Intro_to_AI\project1\part2\train.csv
test : C:\Coding\Intro_to_AI\project1\part2\test.csv

Train size: 1302  cols: ['id', 'content', 'label']
Test  size: 500  cols: ['id', 'content']

Label distribution:
label
1    837
0    465
Name: count, dtype: int64


,id,content,label
0,1166,i think we should consider the actually the cl...,1
1,1127,Can you reset my password? My account password...,1
2,1240,im wondering if we need to no wait the test ca...,1
3,1853,"This code needs to be reviewed by tomorrow, do...",0
4,194,"This homework is taking forever, btw the new s...",0


In [7]:
# Standardize: rename text column to 'text', ensure 'id' exists.
TEXT_COL_CANDIDATES = ['text', 'content', 'utterance', 'sentence']

def standardize(df: pd.DataFrame) -> pd.DataFrame:
    text_col = next((c for c in TEXT_COL_CANDIDATES if c in df.columns), None)
    if text_col is None:
        raise KeyError(f"No text column found in {df.columns.tolist()}")
    if text_col != 'text':
        df = df.rename(columns={text_col: 'text'})
    if 'id' not in df.columns:
        df = df.reset_index(drop=True)
        df['id'] = df.index
    return df

train_df = standardize(train_df)
public_test_df = standardize(public_test_df)

print('Standardization complete.')
print(f"Train cols: {train_df.columns.tolist()}")
print(f"Test  cols: {public_test_df.columns.tolist()}")

if 'label' in train_df.columns:
    counts = train_df['label'].value_counts()
    print(f"\nLabel distribution (train):\n{counts}")
    if counts.min() > 0:
        print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.2f}")


Standardization complete.
Train cols: ['id', 'text', 'label']
Test  cols: ['id', 'text']

Label distribution (train):
label
1    837
0    465
Name: count, dtype: int64
Imbalance ratio (max/min): 1.80


## Part 1: Data Balancing

You must implement and compare two methods:
1. **Basic (Required)**: Random Over-sampling.
2. **Advanced (Choose 1+)**: EDA, Back-translation, SMOTE, or Cost-Sensitive Learning.

In [8]:
def perform_balancing(df: pd.DataFrame, method: str = 'random', random_state: int = 42) -> pd.DataFrame:
    """Phase 2 balancing.

    method='random' -> Random Over-sampling: duplicate minority class samples until 1:1.
    method='none'   -> return the dataframe unchanged.

    Cost-sensitive learning (class_weight='balanced') is handled at model fit time,
    not here, so it can be compared against random over-sampling fairly.
    """
    if method in ('none', None):
        return df.reset_index(drop=True)

    if method == 'random':
        counts = df['label'].value_counts()
        majority_label = counts.idxmax()
        minority_label = counts.idxmin()
        df_majority = df[df['label'] == majority_label]
        df_minority = df[df['label'] == minority_label]

        df_minority_upsampled = resample(
            df_minority,
            replace=True,
            n_samples=len(df_majority),
            random_state=random_state,
        )
        df_balanced = pd.concat([df_majority, df_minority_upsampled], axis=0)
        df_balanced = df_balanced.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
        return df_balanced

    raise ValueError(f"Unknown balancing method: {method}")


# Smoke test
sample_balanced = perform_balancing(train_df, method='random')
print('After Random Over-sampling:')
print(sample_balanced['label'].value_counts())


After Random Over-sampling:
label
0    837
1    837
Name: count, dtype: int64


## Part 2: Baseline Classifier (TF-IDF + SVM)

Establish a baseline. Use Macro-F1 as your primary metric.

In [9]:
# Optional text preprocessing hook. Phase 2 baseline uses raw text directly.
# Reserved as a Phase 3 extension point (lemmatization, stop-word removal, etc.).
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    return text.strip()


In [10]:
def get_model(model_type: str = 'svm', class_weight=None, C: float = 1.0, random_state: int = 42):
    """Phase 2 model factory.

    'svm' returns LinearSVC. Pass class_weight='balanced' for the cost-sensitive variant.
    LinearSVC is faster and more deterministic than kernel SVC on this dataset while
    remaining a Support Vector Machine.
    """
    if model_type == 'svm':
        return LinearSVC(
            C=C,
            class_weight=class_weight,
            random_state=random_state,
            max_iter=5000,
        )
    raise ValueError(f"Unknown model_type: {model_type}")


In [11]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label'],
)

print(f"Train subset : {len(train_data)} samples")
print(f"Val   subset : {len(val_data)} samples")
print('Train label distribution:')
print(train_data['label'].value_counts())
print('Val label distribution:')
print(val_data['label'].value_counts())


Train subset : 1041 samples
Val   subset : 261 samples
Train label distribution:
label
1    669
0    372
Name: count, dtype: int64
Val label distribution:
label
1    168
0     93
Name: count, dtype: int64


In [12]:
# Fair comparison: same TF-IDF + LinearSVC across three settings.
# Same train/val split, same feature config, only the balancing strategy varies.

TFIDF_PARAMS = dict(ngram_range=(1, 2), min_df=2, sublinear_tf=True)


def run_experiment(name: str, train_data, val_data, *, balance: str, class_weight):
    train_used = perform_balancing(train_data, method=balance)

    vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
    X_train = vectorizer.fit_transform(train_used['text'].astype(str))
    y_train = train_used['label'].values

    clf = get_model('svm', class_weight=class_weight)
    clf.fit(X_train, y_train)

    X_val = vectorizer.transform(val_data['text'].astype(str))
    y_val = val_data['label'].values
    y_pred = clf.predict(X_val)

    macro_f1 = f1_score(y_val, y_pred, average='macro')
    print(f"\n=== {name} ===")
    print(f"Train rows used: {len(train_used)}  |  TF-IDF vocab: {len(vectorizer.vocabulary_)}")
    print(classification_report(y_val, y_pred, target_names=['Incomplete(0)', 'Complete(1)']))
    print(f"Macro-F1: {macro_f1:.4f}")
    return {'name': name, 'macro_f1': macro_f1}


results = []
results.append(run_experiment(
    'Baseline (no balancing)',
    train_data, val_data, balance='none', class_weight=None,
))
results.append(run_experiment(
    'Random Over-sampling',
    train_data, val_data, balance='random', class_weight=None,
))
results.append(run_experiment(
    'Cost-Sensitive (class_weight=balanced)',
    train_data, val_data, balance='none', class_weight='balanced',
))

summary = pd.DataFrame([{'method': r['name'], 'macro_f1': r['macro_f1']} for r in results])
print('\n=== Summary ===')
print(summary.to_string(index=False))

best = max(results, key=lambda r: r['macro_f1'])
print(f"\nBest balancing strategy on the validation split: {best['name']}  (macro_f1={best['macro_f1']:.4f})")



=== Baseline (no balancing) ===
Train rows used: 1041  |  TF-IDF vocab: 2168
               precision    recall  f1-score   support

Incomplete(0)       0.78      0.67      0.72        93
  Complete(1)       0.83      0.90      0.86       168

     accuracy                           0.82       261
    macro avg       0.81      0.78      0.79       261
 weighted avg       0.81      0.82      0.81       261

Macro-F1: 0.7919

=== Random Over-sampling ===
Train rows used: 1338  |  TF-IDF vocab: 3028
               precision    recall  f1-score   support

Incomplete(0)       0.80      0.61      0.70        93
  Complete(1)       0.81      0.92      0.86       168

     accuracy                           0.81       261
    macro avg       0.81      0.76      0.78       261
 weighted avg       0.81      0.81      0.80       261

Macro-F1: 0.7777

=== Cost-Sensitive (class_weight=balanced) ===
Train rows used: 1041  |  TF-IDF vocab: 2168
               precision    recall  f1-score   support

## Part 3: Enhancement with K-Fold (Optional)

Improve your results with implement  K-FOLD Cross Validtaion if needed.

In [13]:
# Stratified K-Fold cross-validation is left as a Phase 3 enhancement.
# Phase 2 stops at the three-way balancing comparison above.


## Part 4: Final Submission

Train on the full dataset using your best found configuration and generate `submission.csv`.

In [14]:
# Train the final model on ALL training data using the best configuration.
# Adjust these two knobs based on the comparison above.
BEST_BALANCE_METHOD = 'random'   # 'none' or 'random'
BEST_CLASS_WEIGHT   = None       # None or 'balanced'

final_train_df = perform_balancing(train_df, method=BEST_BALANCE_METHOD)
print(f"Final training size: {len(final_train_df)}")
print(final_train_df['label'].value_counts())

final_vectorizer = TfidfVectorizer(**TFIDF_PARAMS)
X_full = final_vectorizer.fit_transform(final_train_df['text'].astype(str))
y_full = final_train_df['label'].values

final_model = get_model('svm', class_weight=BEST_CLASS_WEIGHT)
final_model.fit(X_full, y_full)
print(f"Final model trained on {X_full.shape[0]} samples; vocab={len(final_vectorizer.vocabulary_)}")


Final training size: 1674
label
0    837
1    837
Name: count, dtype: int64
Final model trained on 1674 samples; vocab=3595


In [15]:
def generate_kaggle_submission(model, vectorizer, test_df, output_name='submission.csv'):
    X_test = vectorizer.transform(test_df['text'].astype(str))
    test_predictions = model.predict(X_test)

    submission = pd.DataFrame({
        'id': test_df['id'].values,
        'label': test_predictions.astype(int),
    })
    submission.to_csv(output_name, index=False)

    print(f"Saved -> {output_name}")
    print('Prediction distribution:')
    print(submission['label'].value_counts())
    display(submission.head(10))
    return submission


submission = generate_kaggle_submission(final_model, final_vectorizer, public_test_df)


Saved -> submission.csv
Prediction distribution:
label
1    319
0    181
Name: count, dtype: int64


,id,label
0,1608,0
1,334,1
2,1779,1
3,1015,1
4,1790,1
5,1635,0
6,358,0
7,1574,1
8,700,1
9,284,1


In [16]:
# Download in Colab
try:
    from google.colab import files
    files.download('submission.csv')
    print("Download started!")
except ImportError:
    print("Not running in Colab — file saved locally as submission.csv")

Not running in Colab — file saved locally as submission.csv
